# Golbal Threshold tuning

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
)


In [2]:
OUTPUT_DIR = Path(
    r"C:\Users\alrazz\Downloads\Combine_single_no_na\Combine_single_no_na"
)


In [3]:
THRESHOLD_START = 0.01
THRESHOLD_END = 0.99
THRESHOLD_STEP = 0.01

In [4]:
MODEL_LABELS = [
    "MT",   # 0
    "LY",   # 1
    "SP",   # 2
    "ID",   # 3
    "NA",   # 4
    "HI",   # 5
    "IN",   # 6
    "OP",   # 7
    "IP",   # 8
    "it",   # 9
    "ne",   # 10
    "sr",   # 11
    "nb",   # 12
    "re",   # 13
    "en",   # 14
    "ra",   # 15
    "dtp",  # 16
    "fi",   # 17
    "lt",   # 18
    "rv",   # 19
    "ob",   # 20
    "rs",   # 21
    "av",   # 22
    "ds",   # 23
    "ed",   # 24
]


In [5]:
AGGREGATION_METHODS = [
"max",
"mean",
"top2_mean",
]

## Load Data

In [6]:
chunk_logits_file = (
OUTPUT_DIR / "dev_chunk_logits.npy"
)

document_ids_file = (
OUTPUT_DIR / "dev_document_ids.npy"
)

document_labels_file = (
OUTPUT_DIR / "dev_document_labels.npy"
)

In [7]:
if not chunk_logits_file.exists():
    raise FileNotFoundError(
        f"Could not find:\n{chunk_logits_file}"
    )


if not document_ids_file.exists():
    raise FileNotFoundError(
        f"Could not find:\n{document_ids_file}"
    )

if not document_labels_file.exists():
    raise FileNotFoundError(
        f"Could not find:\n{document_labels_file}"
    )

In [8]:
print("\nLoading chunk logits...")

chunk_logits = np.load(
chunk_logits_file
)

print(
"Chunk logits shape:",
chunk_logits.shape
)


Loading chunk logits...
Chunk logits shape: (3048, 25)


In [9]:
print("\nLoading document IDs...")

saved_document_ids = np.load(
document_ids_file
)

print(
"Saved document IDs shape:",
saved_document_ids.shape
)


Loading document IDs...
Saved document IDs shape: (609,)


In [10]:
print("\nLoading document labels...")

document_labels = np.load(
document_labels_file
)

print(
"Document labels shape:",
document_labels.shape
)


Loading document labels...
Document labels shape: (609, 25)


In [11]:
# Shape check

if chunk_logits.ndim != 2:
    raise ValueError(
    "Expected chunk logits to be a 2D array "
    "(number_of_chunks, number_of_classes).\n"
    f"Got shape: {chunk_logits.shape}"
)

if document_labels.ndim != 2:
    raise ValueError(
    "Expected document labels to be a 2D array "
    "(number_of_documents, number_of_classes).\n"
    f"Got shape: {document_labels.shape}"
)

num_chunks = chunk_logits.shape[0]

num_classes = chunk_logits.shape[1]

num_documents = document_labels.shape[0]

print("\nNumber of chunks:", num_chunks)
print("Number of documents:", num_documents)
print("Number of classes:", num_classes)

if len(MODEL_LABELS) != num_classes:
    raise ValueError(
    f"\nMODEL_LABELS contains {len(MODEL_LABELS)} labels, "
    f"but logits contain {num_classes} classes."
)



Number of chunks: 3048
Number of documents: 609
Number of classes: 25


In [12]:
from datasets import load_from_disk

dev_dataset = load_from_disk(
    r"C:\Users\alrazz\Documents\GitHub\persian_registers\Register\Dataset_Hugging_face\without_NA\Combined_single_no_NA\dev"
)


c:\Users\alrazz\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
from transformers import (
    AutoTokenizer)

tokenizer = AutoTokenizer.from_pretrained(
    "BAAI/bge-m3-retromae"
)


In [14]:
MAX_LENGTH = 1024
STRIDE = 768
return_overflowing_tokens=True

## Reconstruct Dev Chunk → Document Mapping


In [15]:


from pathlib import Path
import numpy as np
import pandas as pd
import json


# --------------------------------------------------
# Sliding-window settings
# MUST match training exactly
# --------------------------------------------------

MAX_LENGTH = 1024
STRIDE = 768

# --------------------------------------------------
# Load saved outputs
# --------------------------------------------------

chunk_logits_file = (
    OUTPUT_DIR / "dev_chunk_logits.npy"
)

document_ids_file = (
    OUTPUT_DIR / "dev_document_ids.npy"
)

document_labels_file = (
    OUTPUT_DIR / "dev_document_labels.npy"
)

print("=" * 80)
print("LOADING SAVED DEV OUTPUTS")
print("=" * 80)

chunk_logits = np.load(
    chunk_logits_file
)

saved_document_ids = np.load(
    document_ids_file
)

document_labels = np.load(
    document_labels_file
)

print(
    "\nChunk logits shape:",
    chunk_logits.shape
)

print(
    "Document IDs shape:",
    saved_document_ids.shape
)

print(
    "Document labels shape:",
    document_labels.shape
)


LOADING SAVED DEV OUTPUTS

Chunk logits shape: (3048, 25)
Document IDs shape: (609,)
Document labels shape: (609, 25)


In [16]:
# %%
print("\n" + "=" * 80)
print("ORIGINAL DEV DATASET")
print("=" * 80)

print(dev_dataset)

print(
    "\nNumber of original documents:",
    len(dev_dataset)
)

print(
    "\nDataset columns:",
    dev_dataset.column_names
)

print(
    "\nFirst document:"
)

print(
    dev_dataset[0]
)



ORIGINAL DEV DATASET
Dataset({
    features: ['text', 'labels'],
    num_rows: 609
})

Number of original documents: 609

Dataset columns: ['text', 'labels']

First document:
{'text': 'تامین اجتماعی\nکلیه کارمندان دولت اعم از رسمی و پیمانی از تمامی مزایای تامین اجتماعی مانند ؛ بازنشستگی؛ ازکار افتادگی ، فوت ؛ بیمه بیکاری و…. بر خوردار بوده ودر صورتیکه کارمند بخواهد تحت پوشش سایر بیمه ها و صندوق ها قرار گیرد و غیر از سازمان بازنشستگی کشوری و تامین اجتماعی را برگزیند بلامانع میباشد. در صورت نیاز به وکیل تامین اجتماعی با شماره ۰۹۱۲۶۱۶۱۱۲۱تماس حاصل فرمائید.\nبیمه خدمات درمانی\nکلیه شاغلین و بازنشستگان می توانند والدین و فرزندان خود را همرا با خود و همسر تحت پوشش بیمه خدمات درمانی قرار دهند.\nبیمه بیکاری\nبیمه بیکاری صرفا شامل کسانی میگردد که بدون میل و اراده خود بیکار شده باشند.\nتمامی کسانی که مشمول قانون کار و کارکشاورزی بوده و حداقل دارای ۶ ماه پرداخت حق بیمه و تحت پوشش قانون تامین اجتماعی باشند قادر به استفاده از مزایای بیمه بیکاری می گردند اما می بایست حق بیمه بیکاری آنان به میزان ۳ 

In [17]:
print("\n" + "=" * 80)
print("RECONSTRUCTING CHUNK → DOCUMENT MAPPING")
print("=" * 80)

chunk_document_ids = []

chunk_labels = []

# Process documents in batches.
#
# A batch is not necessary for correctness here, but it
# prevents us from creating one enormous tokenizer output
# if the dev set is large.

BATCH_SIZE = 100

for start in range(
    0,
    len(dev_dataset),
    BATCH_SIZE
):

    end = min(
        start + BATCH_SIZE,
        len(dev_dataset)
    )

    batch = dev_dataset[
        start:end
    ]

    texts = batch["text"]

    labels = batch["labels"]

    # Document IDs are exactly the original row positions
    document_ids = list(
        range(start, end)
    )

    tokenized = tokenizer(
        texts,
        truncation=True,
        max_length=MAX_LENGTH,
        stride=STRIDE,
        return_overflowing_tokens=True,
    )

    sample_mapping = tokenized.pop(
        "overflow_to_sample_mapping"
    )

    # Each generated chunk gets the ID of
    # the original document that generated it.

    for sample_idx in sample_mapping:

        chunk_document_ids.append(
            document_ids[sample_idx]
        )

        chunk_labels.append(
            labels[sample_idx]
        )


chunk_document_ids = np.asarray(
    chunk_document_ids,
    dtype=np.int64
)

chunk_labels = np.asarray(
    chunk_labels,
    dtype=np.float32
)

print(
    "\nReconstructed chunk-document IDs:",
    chunk_document_ids.shape
)

print(
    "Reconstructed chunk labels:",
    chunk_labels.shape
)

print(
    "Number of reconstructed chunks:",
    len(chunk_document_ids)
)

print(
    "Number of reconstructed documents:",
    len(np.unique(chunk_document_ids))
)


RECONSTRUCTING CHUNK → DOCUMENT MAPPING

Reconstructed chunk-document IDs: (3048,)
Reconstructed chunk labels: (3048, 25)
Number of reconstructed chunks: 3048
Number of reconstructed documents: 609


In [18]:

print("\n" + "=" * 80)
print("VERIFYING RECONSTRUCTION")
print("=" * 80)

num_saved_chunks = chunk_logits.shape[0]

num_reconstructed_chunks = len(
    chunk_document_ids
)

num_saved_documents = len(
    saved_document_ids
)

num_original_documents = len(
    dev_dataset
)

print(
    "\nSaved chunks:",
    num_saved_chunks
)

print(
    "Reconstructed chunks:",
    num_reconstructed_chunks
)

print(
    "\nSaved documents:",
    num_saved_documents
)

print(
    "Original dev documents:",
    num_original_documents
)


# --------------------------------------------------
# Check 1: chunk count
# --------------------------------------------------

if num_saved_chunks != num_reconstructed_chunks:

    raise ValueError(
        "\nCHUNK COUNT MISMATCH!\n"
        f"Saved logits contain {num_saved_chunks} chunks.\n"
        f"Reconstructed tokenizer output contains "
        f"{num_reconstructed_chunks} chunks.\n\n"
        "This means the tokenization settings or tokenizer "
        "do not exactly reproduce the original training setup."
    )


# --------------------------------------------------
# Check 2: document count
# --------------------------------------------------

if num_saved_documents != num_original_documents:

    raise ValueError(
        "\nDOCUMENT COUNT MISMATCH!\n"
        f"Saved document IDs contain "
        f"{num_saved_documents} documents.\n"
        f"Original dev dataset contains "
        f"{num_original_documents} documents."
    )


# --------------------------------------------------
# Check 3: reconstructed unique document IDs
# --------------------------------------------------

unique_reconstructed_ids = np.unique(
    chunk_document_ids
)

if len(unique_reconstructed_ids) != num_saved_documents:

    raise ValueError(
        "\nUnique reconstructed document count does not "
        "match saved document count."
    )


# --------------------------------------------------
# Check 4: document IDs
# --------------------------------------------------

if not np.array_equal(
    unique_reconstructed_ids,
    saved_document_ids
):

    raise ValueError(
        "\nDOCUMENT ID ORDER MISMATCH!\n\n"
        "The reconstructed document IDs do not match "
        "dev_document_ids.npy."
    )


print(
    "\n✓ Chunk count matches."
)

print(
    "✓ Document count matches."
)

print(
    "✓ Document ID order matches."
)

print(
    "\nRECONSTRUCTION SUCCESSFUL."
)



VERIFYING RECONSTRUCTION

Saved chunks: 3048
Reconstructed chunks: 3048

Saved documents: 609
Original dev documents: 609

✓ Chunk count matches.
✓ Document count matches.
✓ Document ID order matches.

RECONSTRUCTION SUCCESSFUL.


In [19]:
# --------------------------------------------------
# SAVE RECONSTRUCTED CHUNK -> DOCUMENT MAPPING
# --------------------------------------------------

chunk_document_ids_file = (
    OUTPUT_DIR / "dev_chunk_document_ids.npy"
)

np.save(
    chunk_document_ids_file,
    chunk_document_ids,
)

print(
    "\nSaved chunk -> document mapping to:"
)

print(
    chunk_document_ids_file
)

print(
    "Chunk-document ID shape:",
    chunk_document_ids.shape
)



Saved chunk -> document mapping to:
C:\Users\alrazz\Downloads\Combine_single_no_na\Combine_single_no_na\dev_chunk_document_ids.npy
Chunk-document ID shape: (3048,)


## Recover Chunk → Document Mapping

In [20]:
if chunk_document_ids_file.exists():
    print(
    "\nFound:",
    chunk_document_ids_file.name
    )
    chunk_document_ids = np.load(
    chunk_document_ids_file
    )
    print(
        "Chunk document IDs shape:",
        chunk_document_ids.shape
    )
    mapping_source = (
        "dev_chunk_document_ids.npy"
    )
elif len(saved_document_ids) == num_chunks:
    print(
    "\nNo separate chunk-document-ID file found."
    )
    print(
    "Using dev_document_ids.npy as the "
    "chunk → document mapping because its "
    "length matches the number of chunks."
    )
    chunk_document_ids = saved_document_ids
    mapping_source = (
    "dev_document_ids.npy"
    )
else:
    raise RuntimeError(
    "\nCannot perform arbitrary document-level "
    "aggregation with the currently saved files.\n\n"
    f"Chunk logits contain {num_chunks} chunks.\n"
    f"dev_document_ids.npy contains "
    f"{len(saved_document_ids)} IDs.\n\n"
    "Because the training script saved only the "
    "UNIQUE document IDs, the chunk → document "
    "mapping was not preserved.\n\n"
    "You would need a file containing one document "
    "ID per chunk, e.g.\n\n"
    "    dev_chunk_document_ids.npy\n\n"
    "Without that mapping, max/mean/top2_mean "
    "cannot be reconstructed correctly."
    )






Found: dev_chunk_document_ids.npy
Chunk document IDs shape: (3048,)


## Validate Chunk → Document Mapping

In [21]:
if len(chunk_document_ids) != num_chunks:
    raise ValueError(
    "\nNumber of chunk document IDs does not match "
    "number of chunks.\n"
    f"Chunks: {num_chunks}\n"
    f"Document IDs: {len(chunk_document_ids)}"
)


In [22]:
unique_chunk_document_ids = pd.unique(
chunk_document_ids
)

print(
"\nMapping source:",
mapping_source
)

print(
"Unique documents represented by chunks:",
len(unique_chunk_document_ids)
)

print(
"Documents represented by labels:",
num_documents
)


Mapping source: dev_chunk_document_ids.npy
Unique documents represented by chunks: 609
Documents represented by labels: 609


## Verify Document ID Ordering

In [23]:
if len(unique_chunk_document_ids) != num_documents:
    raise ValueError(
    "\nNumber of unique document IDs from chunks "
    "does not match number of document labels.\n"
    f"Chunk-derived documents: "
    f"{len(unique_chunk_document_ids)}\n"
    f"Document labels: {num_documents}"
)

if len(saved_document_ids) == num_documents:
    if not np.array_equal(
    unique_chunk_document_ids,
    saved_document_ids,
):
            raise ValueError(
        "\nDocument ID ordering mismatch.\n\n"
        "The order of documents reconstructed from "
        "the chunk IDs does not match the order of "
        "dev_document_ids.npy.\n\n"
        "Do NOT continue until this is resolved."
    )
print(
    "\nDocument ID ordering verified."
)



Document ID ordering verified.


## LOGITS -> PROBABILITIES

In [24]:
print(
"\nConverting chunk logits to probabilities..."
)

chunk_probabilities = np.empty_like(
chunk_logits,
dtype=np.float64,
)

positive_mask = chunk_logits >= 0

chunk_probabilities[
positive_mask
] = (
1.0
/ (
1.0
+ np.exp(
-chunk_logits[
positive_mask
]
)
)
)

chunk_probabilities[
~positive_mask
] = (
np.exp(
chunk_logits[
~positive_mask
]
)
/ (
1.0
+ np.exp(
chunk_logits[
~positive_mask
]
)
)
)

print(
"Probability range:",
float(chunk_probabilities.min()),
"to",
float(chunk_probabilities.max())
)


Converting chunk logits to probabilities...
Probability range: 0.0018976697465404868 to 0.9922124743461609


## Document-Level Aggregation

In [25]:
def aggregate_chunk_probabilities(
chunk_probabilities,
chunk_document_ids,
aggregation="max",
):
    """
    Aggregate chunk-level probabilities into
    document-level probabilities.
    """
    chunk_document_ids = np.asarray(
    chunk_document_ids
    )
    unique_document_ids = pd.unique(
    chunk_document_ids
    )
    document_probabilities = []
    for document_id in unique_document_ids:
        mask = (
            chunk_document_ids
            == document_id
        )
        document_chunk_probabilities = (
            chunk_probabilities[mask]
        )
        if aggregation == "max":
            document_probability = np.max(
                document_chunk_probabilities,
                axis=0,
            )
        elif aggregation == "mean":
            document_probability = np.mean(
                document_chunk_probabilities,
                axis=0,
            )
        elif aggregation == "top2_mean":
            sorted_probabilities = np.sort(
                document_chunk_probabilities,
                axis=0,
            )
            top_k = min(
                2,
                sorted_probabilities.shape[0],
            )
            document_probability = np.mean(
                sorted_probabilities[
                    -top_k:
                ],
                axis=0,
            )
        else:
            raise ValueError(
                f"Unknown aggregation method: "
                f"{aggregation}"
            )
        document_probabilities.append(
        document_probability
        )
    return (
        np.asarray(unique_document_ids),
        np.asarray(document_probabilities),
    )




## Test All Aggregation Methods

In [26]:
aggregated_probabilities = {}

aggregated_document_ids = {}

In [27]:
for aggregation in AGGREGATION_METHODS:
    print(
        f"\nAggregating using: {aggregation}"
    )
    (
        ids,
        probabilities,
    ) = aggregate_chunk_probabilities(
        chunk_probabilities=chunk_probabilities,
        chunk_document_ids=chunk_document_ids,
        aggregation=aggregation,
    )
    if len(ids) != num_documents:
        raise ValueError(
            f"Aggregation '{aggregation}' produced "
            f"{len(ids)} documents, expected "
            f"{num_documents}."
        )
    if len(saved_document_ids) == num_documents:
        if not np.array_equal(
            ids,
            saved_document_ids,
        ):
            raise ValueError(
                f"Document ID order mismatch for "
                f"aggregation '{aggregation}'."
            )
    aggregated_document_ids[
        aggregation
    ] = ids
    aggregated_probabilities[
        aggregation
    ] = probabilities
    print(
        "Result shape:",
        probabilities.shape
    )




Aggregating using: max
Result shape: (609, 25)

Aggregating using: mean
Result shape: (609, 25)

Aggregating using: top2_mean
Result shape: (609, 25)


## Aggregation Sanity Check

In [28]:
for aggregation in AGGREGATION_METHODS:
    probabilities = (
        aggregated_probabilities[
            aggregation
        ]
    )
    if probabilities.shape != (
        num_documents,
        num_classes,
    ):
        raise ValueError(
        f"Unexpected shape for "
        f"{aggregation}: "
        f"{probabilities.shape}"
    )
print(
"\nAll aggregation methods passed "
"shape checks."
)


All aggregation methods passed shape checks.


## THRESHOLD SEARCH

In [29]:
thresholds = np.arange(
THRESHOLD_START,
THRESHOLD_END
+ THRESHOLD_STEP / 2,
THRESHOLD_STEP,
)

print(
"\nNumber of thresholds:",
len(thresholds)
)

print(
"First threshold:",
thresholds[0]
)

print(
"Last threshold:",
thresholds[-1]
)


Number of thresholds: 99
First threshold: 0.01
Last threshold: 0.99


## FIND BEST THRESHOLD

In [30]:
def calculate_metrics(
y_true,
y_pred,
):
    """
    Calculate the same document-level metrics
    used in the original training script.
    """
    return {
    "macro_f1": f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    ),
        "micro_f1": f1_score(
        y_true,
        y_pred,
        average="micro",
        zero_division=0,
    ),
        "weighted_f1": f1_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0,
    ),
        "macro_precision": precision_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    ),
        "macro_recall": recall_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    ),
        "micro_precision": precision_score(
        y_true,
        y_pred,
        average="micro",
        zero_division=0,
    ),
        "micro_recall": recall_score(
        y_true,
        y_pred,
        average="micro",
        zero_division=0,
    ),
}


## GLOBAL THRESHOLD TUNING

In [31]:
global_results = []

print("\n" + "=" * 90)
print("GLOBAL THRESHOLD TUNING")
print("=" * 90)

for aggregation in AGGREGATION_METHODS:
    print(
    f"\nAggregation: {aggregation}"
)
    document_probabilities = (
    aggregated_probabilities[
        aggregation
    ]
)
    for threshold in thresholds:
        predictions = (
        document_probabilities
        >= threshold
    ).astype(np.int32)
        metrics = calculate_metrics(
        document_labels,
        predictions,
    )
        global_results.append({
            "aggregation": aggregation,
            "threshold": float(
            threshold
        ),
        **metrics,
        })
global_results_df = pd.DataFrame(
global_results
)


GLOBAL THRESHOLD TUNING

Aggregation: max



Aggregation: mean

Aggregation: top2_mean


## Best Global Threshold Per Aggregation

In [32]:
best_global_results = []

for aggregation in AGGREGATION_METHODS:
    aggregation_df = (
    global_results_df[
        global_results_df[
            "aggregation"
        ]
        == aggregation
    ]
)
    best_idx = aggregation_df[
    "macro_f1"
].idxmax()
    best_row = global_results_df.loc[
    best_idx
]
    best_global_results.append(
    best_row.to_dict()
)
best_global_df = pd.DataFrame(
best_global_results
)

In [33]:
print("\n" + "=" * 90)
print("BEST GLOBAL THRESHOLD PER AGGREGATION")
print("=" * 90)

print(
best_global_df[
[
"aggregation",
"threshold",
"macro_f1",
"micro_f1",
"weighted_f1",
"macro_precision",
"macro_recall",
]
].to_string(index=False)
)


BEST GLOBAL THRESHOLD PER AGGREGATION
aggregation  threshold  macro_f1  micro_f1  weighted_f1  macro_precision  macro_recall
        max       0.45  0.726583  0.744034     0.741070         0.705702      0.757887
       mean       0.30  0.756163  0.762279     0.760398         0.745788      0.775876
  top2_mean       0.35  0.734547  0.744641     0.743753         0.693892      0.787920


In [34]:
best_global_file = (
OUTPUT_DIR
/ "best_global_threshold_by_aggregation.csv"
)

best_global_df.to_csv(
best_global_file,
index=False,
)

In [35]:
# Overall Best Global Configuration

overall_best_global_idx = (
global_results_df[
"macro_f1"
].idxmax()
)

overall_best_global = (
global_results_df.loc[
overall_best_global_idx
]
)

print(
"\n" + "=" * 90
)

print(
"OVERALL BEST GLOBAL CONFIGURATION"
)

print(
"=" * 90
)

print(
"\nAggregation:",
overall_best_global[
"aggregation"
]
)

print(
"Threshold:",
overall_best_global[
"threshold"
]
)

print(
"Macro F1:",
overall_best_global[
"macro_f1"
]
)

print(
"Micro F1:",
overall_best_global[
"micro_f1"
]
)

print(
"Weighted F1:",
overall_best_global[
"weighted_f1"
]
)


OVERALL BEST GLOBAL CONFIGURATION

Aggregation: mean
Threshold: 0.3
Macro F1: 0.7561626270057852
Micro F1: 0.762278978388998
Weighted F1: 0.7603983206554047


## Save Overall Best Global Configuration

In [36]:
best_global_json = {
    "method": "global_threshold",
    "aggregation": str(
    overall_best_global[
        "aggregation"
    ]
),
"threshold": float(
    overall_best_global[
        "threshold"
    ]
),
"macro_f1": float(
    overall_best_global[
        "macro_f1"
    ]
),
"micro_f1": float(
    overall_best_global[
        "micro_f1"
    ]
),
"weighted_f1": float(
    overall_best_global[
        "weighted_f1"
    ]
),
"macro_precision": float(
    overall_best_global[
        "macro_precision"
    ]
),
"macro_recall": float(
    overall_best_global[
        "macro_recall"
    ]
),
"micro_precision": float(
    overall_best_global[
        "micro_precision"
    ]
),
"micro_recall": float(
    overall_best_global[
        "micro_recall"
    ]
),
}

In [37]:
with open(
OUTPUT_DIR
/ "best_global_configuration.json",
"w",
encoding="utf-8",
) as f:
    json.dump(
    best_global_json,
    f,
    indent=2,
)


## PER-CLASS PERFORMANCE AT BEST GLOBAL THRESHOLD

In [38]:
global_per_class_results = []

for aggregation in AGGREGATION_METHODS:
    aggregation_df = (
    global_results_df[
        global_results_df[
            "aggregation"
        ]
        == aggregation
    ]
)
    best_idx = aggregation_df[
    "macro_f1"
].idxmax()
    best_row = global_results_df.loc[
    best_idx
]
    threshold = float(
    best_row["threshold"]
)
    probabilities = (
    aggregated_probabilities[
        aggregation
    ]
)
    predictions = (
    probabilities
    >= threshold
).astype(np.int32)
    per_class_f1 = f1_score(
    document_labels,
    predictions,
    average=None,
    zero_division=0,
)
    per_class_precision = precision_score(
    document_labels,
    predictions,
    average=None,
    zero_division=0,
)
    per_class_recall = recall_score(
    document_labels,
    predictions,
    average=None,
    zero_division=0,
)
    actual_positives = (
    document_labels.sum(
        axis=0
    )
)
    predicted_positives = (
    predictions.sum(
        axis=0
    )
)
    for class_idx in range(
    num_classes
):
            global_per_class_results.append({
                  "aggregation": aggregation,
                  "global_threshold": threshold,
                  "class_index": class_idx,
                  "label": MODEL_LABELS[
                        class_idx
                    ],
                    "f1": per_class_f1[
                        class_idx
                    ],
                    "precision": per_class_precision[
                        class_idx
                    ],
                    "recall": per_class_recall[
                        class_idx
                    ],
                    "actual_positives": int(
                        actual_positives[
                        class_idx
                    ]
                ),
                    "predicted_positives": int(
                        predicted_positives[
                        class_idx
                    ]
                ),
                    })
global_per_class_df = pd.DataFrame(
global_per_class_results
)

In [39]:
global_per_class_file = (
OUTPUT_DIR
/ "global_threshold_per_class_all_aggregations.csv"
)

global_per_class_df.to_csv(
global_per_class_file,
index=False,
)

# Per-class Threshold Tuning

In [40]:
per_class_results = []

per_class_threshold_dicts = {}

print("\n" + "=" * 90)
print("PER-CLASS THRESHOLD TUNING")
print("=" * 90)


PER-CLASS THRESHOLD TUNING


In [41]:
for aggregation in AGGREGATION_METHODS:
    print(
    f"\nAggregation: {aggregation}"
)
    document_probabilities = (
    aggregated_probabilities[
        aggregation
    ]
)
    optimal_thresholds = []
    for class_idx in range(
    num_classes
):
        label_name = MODEL_LABELS[
        class_idx
    ]
        y_true = (
        document_labels[
            :,
            class_idx
        ]
    )
        y_prob = (
        document_probabilities[
            :,
            class_idx
        ]
    )
        best_f1 = -1.0
        best_threshold = 0.5
        best_precision = 0.0
        best_recall = 0.0
        for threshold in thresholds:
            y_pred = (
            y_prob
            >= threshold
        ).astype(np.int32)
            f1 = f1_score(
            y_true,
            y_pred,
            zero_division=0,
        )
            precision = precision_score(
            y_true,
            y_pred,
            zero_division=0,
        )
            recall = recall_score(
            y_true,
            y_pred,
            zero_division=0,
        )
            if f1 > best_f1:
                best_f1 = f1
                best_threshold = (
                float(threshold)
            )
                best_precision = (
                precision
            )
                best_recall = (
                recall
            )
            actual_positives = int(
        y_true.sum()
    )
            predictions_at_best = (
        y_prob
        >= best_threshold
    ).astype(np.int32)
            predicted_positives = int(
        predictions_at_best.sum()
    )
            optimal_thresholds.append(
        best_threshold
    )
            per_class_results.append({
                "aggregation": aggregation,
                 "class_index": class_idx,
                 "label": label_name,
                 "optimal_threshold":
            best_threshold,
            "f1": best_f1,
            "precision":
            best_precision,
            "recall":
            best_recall,
            "actual_positives":
            actual_positives,
            "predicted_positives":
            predicted_positives,
        })
        print(
        f"{class_idx:3d} | "
        f"{label_name:5s} | "
        f"threshold="
        f"{best_threshold:.2f} | "
        f"F1="
        f"{best_f1:.4f} | "
        f"P="
        f"{best_precision:.4f} | "
        f"R="
        f"{best_recall:.4f}"
        )
    per_class_threshold_dicts[
    aggregation
] = {
    MODEL_LABELS[i]:
        float(
            optimal_thresholds[i]
        )
        for i in range(
        num_classes
    )
}




Aggregation: max
  0 | MT    | threshold=0.85 | F1=0.7500 | P=0.8400 | R=0.6774
  1 | LY    | threshold=0.36 | F1=0.9053 | P=0.8958 | R=0.9149
  2 | SP    | threshold=0.78 | F1=0.8500 | P=0.8500 | R=0.8500
  3 | ID    | threshold=0.42 | F1=0.8099 | P=0.7903 | R=0.8305
  4 | NA    | threshold=0.49 | F1=0.7989 | P=0.7525 | R=0.8514
  5 | HI    | threshold=0.27 | F1=0.7717 | P=0.7424 | R=0.8033
  6 | IN    | threshold=0.51 | F1=0.8217 | P=0.7879 | R=0.8585
  7 | OP    | threshold=0.27 | F1=0.7120 | P=0.6433 | R=0.7971
  8 | IP    | threshold=0.53 | F1=0.7713 | P=0.7611 | R=0.7818
  9 | it    | threshold=0.71 | F1=0.8182 | P=0.7941 | R=0.8438
 10 | ne    | threshold=0.49 | F1=0.7407 | P=0.7229 | R=0.7595
 11 | sr    | threshold=0.16 | F1=0.8861 | P=0.7955 | R=1.0000
 12 | nb    | threshold=0.61 | F1=0.7308 | P=0.7917 | R=0.6786
 13 | re    | threshold=0.24 | F1=0.8846 | P=0.8846 | R=0.8846
 14 | en    | threshold=0.45 | F1=0.7692 | P=0.6667 | R=0.9091
 15 | ra    | threshold=0.69 | F1=0.8

In [42]:
per_class_results_df = pd.DataFrame(
per_class_results
)

## Save Per-Class Thresholds

In [43]:
per_class_threshold_file = (
OUTPUT_DIR
/ "per_class_thresholds_all_aggregations.csv"
)

per_class_results_df.to_csv(
per_class_threshold_file,
index=False,
)

print(
"\nSaved:"
)

print(
per_class_threshold_file
)


Saved:
C:\Users\alrazz\Downloads\Combine_single_no_na\Combine_single_no_na\per_class_thresholds_all_aggregations.csv


In [44]:
#as Json

for aggregation in AGGREGATION_METHODS:
    json_file = (
    OUTPUT_DIR
    / f"per_class_thresholds_{aggregation}.json"
)
    with open(
    json_file,
    "w",
    encoding="utf-8",
) as f:
            json.dump(
        per_class_threshold_dicts[
            aggregation
        ],
        f,
        indent=2,
        ensure_ascii=False,
    )


## Evaluate Per-Class Thresholds

In [45]:
per_class_summary_results = []
per_class_predictions_by_aggregation = {}

for aggregation in AGGREGATION_METHODS:
    document_probabilities = (
    aggregated_probabilities[
        aggregation
    ]
)
    optimal_thresholds = np.asarray([
            per_class_threshold_dicts[
        aggregation
    ][label]
        for label in MODEL_LABELS
    ])
    predictions = (
    document_probabilities
    >= optimal_thresholds
).astype(np.int32)
    per_class_predictions_by_aggregation[
    aggregation
] = predictions
    metrics = calculate_metrics(
    document_labels,
    predictions,
)
    per_class_summary_results.append({
            "aggregation": aggregation,
            "threshold_method":
                "per_class",
            **metrics,
        })
per_class_summary_df = pd.DataFrame(
per_class_summary_results
)


## Save Per-Class Summary

In [46]:
per_class_summary_file = (
OUTPUT_DIR
/ "per_class_threshold_summary_all_aggregations.csv"
)

per_class_summary_df.to_csv(
per_class_summary_file,
index=False,
)

In [47]:
print(
"\n" + "=" * 90
)

print(
"PER-CLASS THRESHOLD RESULTS"
)

print(
"=" * 90
)

print(
per_class_summary_df.to_string(
index=False
)
)


PER-CLASS THRESHOLD RESULTS
aggregation threshold_method  macro_f1  micro_f1  weighted_f1  macro_precision  macro_recall  micro_precision  micro_recall
        max        per_class  0.572760  0.498808     0.582566         0.442319      0.911548         0.341623      0.923913
       mean        per_class  0.588916  0.517361     0.600848         0.462611      0.897547         0.361261      0.911005
  top2_mean        per_class  0.578234  0.502051     0.585948         0.456638      0.892968         0.346015      0.914402


## Per-Class Threshold Detailed Performance

In [48]:
per_class_final_results = []

for aggregation in AGGREGATION_METHODS:
    predictions = (
    per_class_predictions_by_aggregation[
        aggregation
    ]
)
    f1_values = f1_score(
    document_labels,
    predictions,
    average=None,
    zero_division=0,
)
    precision_values = precision_score(
    document_labels,
    predictions,
    average=None,
    zero_division=0,
)
    recall_values = recall_score(
    document_labels,
    predictions,
    average=None,
    zero_division=0,
)
    actual_positives = (
    document_labels.sum(
        axis=0
    )
)
    predicted_positives = (
    predictions.sum(
        axis=0
    )
)
    thresholds_for_aggregation = (
    per_class_threshold_dicts[
        aggregation
    ]
)
    for class_idx in range(
    num_classes
):
        label = MODEL_LABELS[
            class_idx
        ]
        per_class_final_results.append({
                        "aggregation": aggregation,
                        "class_index": class_idx,
                        "label": label,
                        "threshold":
                            thresholds_for_aggregation[
                                label
                        ],
                        "f1":
                            f1_values[
                                class_idx
                        ],
                        "precision":
                            precision_values[
                                class_idx
                        ],
                        "recall":
                            recall_values[
                                class_idx
                        ],
                        "actual_positives":
                            int(
                                actual_positives[
                                    class_idx
                                ]
                            ),
                        "predicted_positives":
                             int(
                                predicted_positives[
                                    class_idx
                                ]               
                            ),
                        })
per_class_final_df = pd.DataFrame(
per_class_final_results
)








In [49]:
per_class_final_file = (
OUTPUT_DIR
/ "per_class_final_performance_all_aggregations.csv"
)

per_class_final_df.to_csv(
per_class_final_file,
index=False,
)

## Compare All Methods

In [50]:
comparison_results = []

for aggregation in AGGREGATION_METHODS:
    document_probabilities = (
    aggregated_probabilities[
        aggregation
    ]
)
# fixed 0.50
    fixed_predictions = (
    document_probabilities
    >= 0.50
).astype(np.int32)
    fixed_metrics = calculate_metrics(
    document_labels,
    fixed_predictions,
)
    comparison_results.append({
            "aggregation": aggregation,
            "threshold_method":
                "fixed_global_0.50",
            "threshold":
                0.50,
            **fixed_metrics,
        })
#Global
    aggregation_df = (
    global_results_df[
        global_results_df[
            "aggregation"
        ]
        == aggregation
    ]
)
    best_idx = aggregation_df[
    "macro_f1"
].idxmax()
    best_global_row = (
    global_results_df.loc[
        best_idx
    ]
)
    comparison_results.append({
            "aggregation": aggregation,
            "threshold_method":
                "optimized_global",
            "threshold":
                float(
                    best_global_row[
                        "threshold"
                    ]
                ),
            "macro_f1":
                 float(
                    best_global_row[
                        "macro_f1"
                    ]
                ),
            "micro_f1":
                float(
                    best_global_row[
                        "micro_f1"
                    ]
                ),
            "weighted_f1":
                float(
                    best_global_row[
                        "weighted_f1"
                    ]
                ),
            "macro_precision":
                float(
                    best_global_row[
                        "macro_precision"
                    ]
                ),
            "macro_recall":
                float(
                    best_global_row[
                        "macro_recall"
                    ]
                ),
            "micro_precision":
                float(
                    best_global_row[
                        "micro_precision"
                    ]
                ),
            "micro_recall":
                float(
                    best_global_row[
                        "micro_recall"
                    ]
                ),
        })
    #Per-class thresholds
    per_class_row = (
    per_class_summary_df[
        per_class_summary_df[
            "aggregation"
        ]
        == aggregation
    ].iloc[0]
)
    comparison_results.append({
        "aggregation": aggregation,
        "threshold_method":
            "optimized_per_class",
        "threshold":
            "multiple",
        **{
            metric: float(
                per_class_row[
                    metric
                ]
            )
            for metric in [
            "macro_f1",
            "micro_f1",
            "weighted_f1",
            "macro_precision",
            "macro_recall",
            "micro_precision",
            "micro_recall",
        ]
    },
    })
comparison_df = pd.DataFrame(
comparison_results
)





## Save Complete Comparison

In [51]:
comparison_file = (
OUTPUT_DIR
/ "aggregation_threshold_comparison.csv"
)

comparison_df.to_csv(
comparison_file,
index=False,
)

print(
"\n" + "=" * 90
)

print(
"AGGREGATION + THRESHOLD COMPARISON"
)

print(
"=" * 90
)

print(
comparison_df[
[
"aggregation",
"threshold_method",
"threshold",
"macro_f1",
"micro_f1",
"weighted_f1",
]
].to_string(
index=False
)
)


AGGREGATION + THRESHOLD COMPARISON
aggregation    threshold_method threshold  macro_f1  micro_f1  weighted_f1
        max   fixed_global_0.50       0.5  0.723839  0.744496     0.741183
        max    optimized_global      0.45  0.726583  0.744034     0.741070
        max optimized_per_class  multiple  0.572760  0.498808     0.582566
       mean   fixed_global_0.50       0.5  0.723679  0.746686     0.738593
       mean    optimized_global       0.3  0.756163  0.762279     0.760398
       mean optimized_per_class  multiple  0.588916  0.517361     0.600848
  top2_mean   fixed_global_0.50       0.5  0.724483  0.743326     0.738135
  top2_mean    optimized_global      0.35  0.734547  0.744641     0.743753
  top2_mean optimized_per_class  multiple  0.578234  0.502051     0.585948


## Find Overall Best Configuration

In [52]:
best_comparison_idx = (
comparison_df[
"macro_f1"
].idxmax()
)

best_comparison = (
comparison_df.loc[
best_comparison_idx
]
)

In [53]:
print(
"\n" + "=" * 90
)

print(
"OVERALL BEST POST-HOC DEV CONFIGURATION"
)

print(
"=" * 90
)

print(
"\nAggregation:",
best_comparison[
"aggregation"
]
)

print(
"Threshold method:",
best_comparison[
"threshold_method"
]
)

print(
"Threshold:",
best_comparison[
"threshold"
]
)

print(
"Macro F1:",
best_comparison[
"macro_f1"
]
)

print(
"Micro F1:",
best_comparison[
"micro_f1"
]
)

print(
"Weighted F1:",
best_comparison[
"weighted_f1"
]
)


OVERALL BEST POST-HOC DEV CONFIGURATION

Aggregation: mean
Threshold method: optimized_global
Threshold: 0.3
Macro F1: 0.7561626270057852
Micro F1: 0.762278978388998
Weighted F1: 0.7603983206554047


## Save Overall Best Configuration

In [54]:
best_configuration = {
    "selection_metric":
    "macro_f1",
    "aggregation":
    str(
        best_comparison[
            "aggregation"
        ]
    ),
    "threshold_method":
    str(
        best_comparison[
            "threshold_method"
        ]
    ),
    "threshold":
    (
        None
        if best_comparison[
            "threshold"
        ] == "multiple"
        else float(
            best_comparison[
                "threshold"
            ]
        )
    ),
    "macro_f1":
    float(
        best_comparison[
            "macro_f1"
        ]
    ),
    "micro_f1":
    float(
        best_comparison[
            "micro_f1"
        ]
    ),
    "weighted_f1":
    float(
        best_comparison[
            "weighted_f1"
        ]
    ),
    "macro_precision":
    float(
        best_comparison[
            "macro_precision"
        ]
    ),
    "macro_recall":
    float(
        best_comparison[
            "macro_recall"
        ]
    ),
    "micro_precision":
    float(
        best_comparison[
            "micro_precision"
        ]
    ),
    "micro_recall":
    float(
        best_comparison[
            "micro_recall"
        ]
    ),
}

In [55]:
with open(
OUTPUT_DIR
/ "best_posthoc_configuration.json",
"w",
encoding="utf-8",
) as f:
    json.dump(
    best_configuration,
    f,
    indent=2,
)


## Global Threshold Curves

In [56]:
fig, ax = plt.subplots(
figsize=(11, 7)
)

for aggregation in AGGREGATION_METHODS:
    aggregation_df = (
    global_results_df[
        global_results_df[
            "aggregation"
        ]
        == aggregation
    ]
)
    ax.plot(
    aggregation_df[
        "threshold"
    ],
    aggregation_df[
        "macro_f1"
    ],
    linewidth=2,
    label=aggregation,
)
ax.set_xlabel(
"Global Threshold"
)

ax.set_ylabel(
"Macro F1"
)

ax.set_title(
"Document-Level Macro F1 vs Global Threshold"
)

ax.grid(
True,
alpha=0.3,
)

ax.legend()

plt.tight_layout()

global_plot_file = (
OUTPUT_DIR
/ "global_threshold_macro_f1_all_aggregations.png"
)

plt.savefig(
global_plot_file,
dpi=300,
bbox_inches="tight",
)

plt.close()

## Per-Class F1 Curves

In [57]:
for aggregation in AGGREGATION_METHODS:
    document_probabilities = (
    aggregated_probabilities[
        aggregation
    ]
)
    fig, ax = plt.subplots(
    figsize=(16, 9)
)
    for class_idx in range(
    num_classes
):
        y_true = (
        document_labels[
            :,
            class_idx
        ]
    )
        y_prob = (
        document_probabilities[
            :,
            class_idx
        ]
    )
        class_f1_scores = []
        for threshold in thresholds:
            y_pred = (
            y_prob
            >= threshold
        ).astype(np.int32)
            f1 = f1_score(
            y_true,
            y_pred,
            zero_division=0,
        )
            class_f1_scores.append(
            f1
        )
        ax.plot(
        thresholds,
        class_f1_scores,
        linewidth=1.5,
        alpha=0.7,
        label=MODEL_LABELS[
            class_idx
        ],
    )
    ax.set_xlabel(
    "Threshold"
)
    ax.set_ylabel(
    "F1"
)
    ax.set_title(
    f"Per-Class F1 vs Threshold "
    f"({aggregation} aggregation)"
)
    ax.grid(
    True,
    alpha=0.3,
)
    ax.legend(
    title="Class",
    loc="center left",
    bbox_to_anchor=(
        1.02,
        0.5
    ),
    fontsize=9,
    title_fontsize=10,
    frameon=True,
)
    plt.tight_layout(
    rect=[
        0,
        0,
        0.82,
        1
    ]
)
    plot_file = (
    OUTPUT_DIR
    / f"per_class_threshold_f1_{aggregation}.png"
)
    plt.close()


## Save Aggregated Document Probabilities

In [58]:
np.save(
    OUTPUT_DIR
    / f"dev_document_probabilities_{aggregation}.npy",
    aggregated_probabilities[
        aggregation
    ],
)


## Save Final Summary

In [59]:
summary = {
    "num_chunks":
    int(num_chunks),
    "num_documents":
    int(num_documents),
    "num_classes":
    int(num_classes),
    "aggregation_methods":
    AGGREGATION_METHODS,
    "threshold_start":
    float(THRESHOLD_START),
    "threshold_end":
    float(THRESHOLD_END),
    "threshold_step":
    float(THRESHOLD_STEP),
    "selection_metric":
    "macro_f1",
    "best_global_configuration":
    best_global_json,
    "best_posthoc_configuration":
    best_configuration,
}

In [60]:
with open(
OUTPUT_DIR
/ "posthoc_tuning_summary.json",
"w",
encoding="utf-8",
) as f:
    json.dump(
    summary,
    f,
    indent=2,
)


In [61]:
print(
"\n" + "=" * 90
)

print(
"POST-HOC TUNING COMPLETE"
)

print(
"=" * 90
)

print(
"\nBest global configuration:"
)

print(
" Aggregation:",
best_global_json[
"aggregation"
]
)

print(
" Threshold:",
best_global_json[
"threshold"
]
)

print(
" Macro F1:",
best_global_json[
"macro_f1"
]
)

print(
"\nBest overall configuration:"
)

print(
" Aggregation:",
best_configuration[
"aggregation"
]
)

print(
" Threshold method:",
best_configuration[
"threshold_method"
]
)

print(
" Threshold:",
best_configuration[
"threshold"
]
)

print(
" Macro F1:",
best_configuration[
"macro_f1"
]
)

print(
"\nAll results saved to:"
)

print(
OUTPUT_DIR
)

print(
"\nIMPORTANT:"
)

print(
"The selected configuration is based ONLY on "
"the Dev set. It must be frozen before evaluating "
"the Test set."
)


POST-HOC TUNING COMPLETE

Best global configuration:
 Aggregation: mean
 Threshold: 0.3
 Macro F1: 0.7561626270057852

Best overall configuration:
 Aggregation: mean
 Threshold method: optimized_global
 Threshold: 0.3
 Macro F1: 0.7561626270057852

All results saved to:
C:\Users\alrazz\Downloads\Combine_single_no_na\Combine_single_no_na

IMPORTANT:
The selected configuration is based ONLY on the Dev set. It must be frozen before evaluating the Test set.
